In [23]:
#import necessary libararies
#process pdfs
from pypdf import PdfReader
from pathlib import Path
import glob
import numpy as np
import os

#create embeddings
from sentence_transformers import SentenceTransformer

#load embeddings
import chromadb

#language model
import ollama

In [2]:
folder_path = r"C:\Users\Sumed\Desktop\rag_assistant\data\*pdf"
pdf_files = glob.glob(folder_path)

if not pdf_files:
        raise Exception('No document(s) found')
else:
    print(f'{len(pdf_files)} document(s) found')

for i in pdf_files:
    print(i)

2 document(s) found
C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf


In [3]:
#some insights about the PDFs in the folder

page_counter = 0
total_words = 0
words_per_page = []
average_words_per_page = 0

for i in pdf_files:
    print('processing file : ', i)
    reader = PdfReader(i)
    max_words_per_page = 0

    for _,j in enumerate(reader.pages):
        current_page = j.extract_text()
        total_words_in_page = current_page.split()
        
        if max_words_per_page < len(total_words_in_page):
            max_words_per_page = len(total_words_in_page)
        words_per_page.append(len(total_words_in_page))

    print('max words in a page :',max_words_per_page) 
    print('total words in the PDF :',sum(words_per_page))
    print('average number of words per page: ', round(np.divide( sum(words_per_page),len(words_per_page) ),0) )
    print('\n')

processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
max words in a page : 1683
total words in the PDF : 49540
average number of words per page:  266.0


processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf
max words in a page : 596
total words in the PDF : 95328
average number of words per page:  298.0




In [4]:
# create chunks
all_chunks = []
chunks = {}
for i in pdf_files:
    print('processing file : ', i)
    reader = PdfReader(i)
    file_name = os.path.basename(i)
    page_counter = 0

    for _,j in enumerate(reader.pages):
        current_page = j.extract_text()
        chunks = {'source_file': file_name
                 ,'page_number': page_counter
                 ,'chunk_id'   : f'{file_name}_{page_counter}'
                 ,'text'       : current_page
        }
        page_counter+=1
        all_chunks.append(chunks)
    
print(f'sample chunks : {all_chunks[:5]}')
        

processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf
sample chunks : [{'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 0, 'chunk_id': 'encoder_decoder_neural_networks.pdf_0', 'text': 'ENCODER-DECODER\nNEURAL NETWORKS\nNal Kalchbrenner'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 1, 'chunk_id': 'encoder_decoder_neural_networks.pdf_1', 'text': 'Encoder-Decoder Neural Networks\nNal Kalchbrenner\nNew College\nUniversity of Oxford\nA thesis submitted for the degree of\nDoctor of Philosophy\nMichaelmas 2017'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 2, 'chunk_id': 'encoder_decoder_neural_networks.pdf_2', 'text': 'To my parents Metka and Bojan'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 3, 'chunk_id': 'encoder_decoder_neural_networks.pdf_3', 'text': '

In [5]:
#create embeddings
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
text_to_embed = [i['text'] for i in all_chunks]
embeddings = embedding_model.encode(text_to_embed)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [20]:
#initialize chromadb with the persistent type so our vectors dont disappear
client = chromadb.PersistentClient(path=r'C:\Users\Sumed\Desktop\rag_assistant\chromadb_vectors\vectors')
collection = client.get_or_create_collection(name = 'vector_storage')

ids            = [i['chunk_id'] for i in all_chunks]
documents      = [i['text'] for i in all_chunks]
embedding_list = embeddings.tolist()
metadata       = [{'source_file': i['source_file'], 'page_number': i['page_number']} for i in all_chunks]

collection.add(ids       = ids
              ,documents = documents
              ,metadatas = metadata 
              ,embeddings = embedding_list  
           )

In [28]:
#initialize ollama and qwen3 with 4 billion parameters
response = ollama.chat(
    model="qwen3:4b",
    messages=[
        {
            "role": "user",
            "content": "What is backpropagation?"
        }
    ]
)
print(response)

model='qwen3:4b' created_at='2026-06-25T17:09:01.5058455Z' done=True done_reason='stop' total_duration=33892021100 load_duration=443986500 prompt_eval_count=16 prompt_eval_duration=48149000 eval_count=2326 eval_duration=33150017999 message=Message(role='assistant', content='**Backpropagation** (short for *backward propagation of errors*) is a **fundamental algorithm** used to efficiently compute the gradients of a neural network\'s loss function with respect to its weights and biases. These gradients are then used by optimization algorithms (like Gradient Descent) to update the network\'s parameters and minimize the loss. Here’s a clear, step-by-step explanation:\n\n---\n\n### 🔑 **Why is Backpropagation Important?**\nWithout backpropagation, training deep neural networks would be **computationally infeasible**. Here’s why:\n- **Problem**: For a neural network with *n* layers, directly calculating gradients (via manual chain rule) would require computing the derivative of the loss funct